#IMPLEMENTACIÓN DE UN CHATBOT PARA CRUCEROS

##Librerías

*Importar* las librerías necesarias para el procesamiento de lenguaje natural

In [1]:
import string
import random
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
nltk.download('punkt') # -- Instalar módulo punkt
nltk.download('wordnet') # -- Instalar módulo wordnet
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

##1 Definición del CORPUS

In [2]:
archivo = open('Corpus_crucero.txt','r', encoding='latin-1', errors='ignore')
raw = archivo.read()

##2 Preprocesamiento del Texto

In [3]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

###2a Preprocesamiento del Texto con NLTK CORPUS

In [4]:
raw=raw.lower() # -- convertir a minúsculas

sent_tokens = nltk.sent_tokenize(raw) # -- Convierte el CORPUS a una lista de sentencias
word_tokens = nltk.word_tokenize(raw) # -- Convierte el CORPUS a una lista de palabras
lemmer = nltk.stem.WordNetLemmatizer() # -- Crear objeto para lematizar

# -- Función para generar lista de tokens lematizados
def LemTokens(tokens):
  return [lemmer.lemmatize(token) for token in tokens]

# -- Recuperar como un diccionario los signos de puntuación a remover
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
# -- Función de lematización removiendo los signos de puntuación
def LemNormalize(text):
  return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))

###2b Preprocesamiento del Texto con NLTK Mensaje de usuario

In [5]:
def preprocesamiento_texto_usuario(user_response):
  # robo_response = ''
  sent_tokens.append(user_response) # -- Añade al final del CORPUS la respuesta del usuario
  TfidfVec = TfidfVectorizer(tokenizer=LemNormalize, stop_words = stopwords.words('spanish'))
  tfidf = TfidfVec.fit_transform(sent_tokens)
  return tfidf

##3 Evaluar similitud MENSAJE USUARIO - CORPUS

In [6]:
# -- Función para determinar la similitud del texto insertado y el CORPUS
def response(user_response):
  robo_response = ''
  tfidf = preprocesamiento_texto_usuario(user_response)
  # 3 Evaluar similitud de coseno entre mensaje de usuario (tfidf[-1]) y el CORPUS (tfidf)
  vals = cosine_similarity(tfidf[-1], tfidf)
  print(vals)
  idx = vals.argsort()[0][-2]
  print(idx)
  flat = vals.flatten()
  print(flat)
  flat.sort()
  print(flat)
  req_tfidf = flat[-2]

  if (req_tfidf == 0):
    robo_response = robo_response+"Lo siento, no te entendí. Póngase en contacto con el personal asistencial"
  else:
    robo_response = robo_response + sent_tokens[idx]
  return robo_response


##4 Definición de coincidencias manual

In [7]:
SALUDOS_INPUTS = ("hola","buenas", "saludos", "qué tal", "hey", "buenos días")
SALUDOS_OUTPUTS = ["Hola", "Hola, ¿Qué tal?", "Hola, ¿Cómo te puedo ayudar?", "Hola, encantado de hablar contigo"]

def saludos(sentence):
  for word in sentence.split():
    if word.lower() in SALUDOS_INPUTS:
      return random.choice(SALUDOS_OUTPUTS)

##5 Generación de respuesta

Programa principal

In [8]:
# -- PROGRAMA PRINCIPAL
flag = True
print('Hola, contestaré a tus preguntas acerca de tus vacaciones en el crucero.')
print('Si ya no deseas continuar, escribe salir, adios o chau')

while (flag == True):
  user_response = input()
  user_response = user_response.lower()

  if (user_response != 'salir') and (user_response != 'adios') and (user_response != 'chau'):
    if (user_response == 'gracias' or user_response == 'muchas gracias'):
      flag = True
      print('No hay de qué')
    else:
      if (saludos(user_response) != None):
        print(saludos(user_response))
      else:
        print(response(user_response))
        sent_tokens.remove(user_response)
  else:
    flag = False
    print('Nos vemos pronto, ¡Cuídate!')

Hola, contestaré a tus preguntas acerca de tus vacaciones en el crucero.
Si ya no deseas continuar, escribe salir, adios o chau
Hola
Hola, ¿Cómo te puedo ayudar?
El crucero ofrece servicios médicos?


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[[0.         0.         0.         0.05333358 0.         0.
  0.         0.         0.         0.         0.         0.20378621
  0.         0.         0.         0.09329546 0.         0.11259635
  0.         0.         0.04629346 0.         0.         0.
  0.         0.         0.         0.         0.04690444 0.
  0.09907999 0.05228097 1.        ]]
11
[0.         0.         0.         0.05333358 0.         0.
 0.         0.         0.         0.         0.         0.20378621
 0.         0.         0.         0.09329546 0.         0.11259635
 0.         0.         0.04629346 0.         0.         0.
 0.         0.         0.         0.         0.04690444 0.
 0.09907999 0.05228097 1.        ]
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.04629346 0.04690444 0.05228097 0.05333358 0.09329546 0.0990